# Lane Theory — 03: Individual Diagnosis (v3)
Loads trained artifacts from Notebook 02, applies the v3 feature
engineering to individual swimmer data, and produces a 6-Pillar
diagnostic report plus a lever-based optimization simulation. No ranking
anywhere — every comparison is against the elite range for the
swimmer's own gender.

In [1]:
import pandas as pd
import numpy as np
import joblib
import pickle
import sys

sys.path.insert(0, "../src")
from feature_engineering import engineer_features
from optimization import (
    primitive_cols, feature_cols, PILLARS,
    compute_pillar_report, print_pillar_report, find_biggest_gap_pillar,
    diagnose_swimmer, print_report,
)

pd.set_option("display.max_columns", None)

models = joblib.load("../models/gender_models.joblib")
with open("../models/shap_values_dict.pkl", "rb") as f:
    shap_values_dict = pickle.load(f)

elite_features = pd.read_csv("../data/elite/processed/100m_freestyle_features.csv")

print(f"Loaded {len(models)} models, SHAP for {len(shap_values_dict)} genders, "
      f"{len(elite_features)} elite rows")

Loaded 2 models, SHAP for 2 genders, 40 elite rows


## Auto-ID generation (for future users)

In [2]:
def get_next_athlete_id(existing_df=None):
    if existing_df is None or existing_df.empty:
        return "U001"
    existing_nums = existing_df["athlete_id"].str.extract(r"U(\d+)")[0].astype(int)
    next_num = existing_nums.max() + 1
    return f"U{next_num:03d}"

def get_next_race_id(athlete_id, existing_df=None):
    if existing_df is None or existing_df.empty:
        return f"{athlete_id}-R01"
    athlete_races = existing_df[existing_df["athlete_id"] == athlete_id]
    if athlete_races.empty:
        return f"{athlete_id}-R01"
    existing_nums = athlete_races["race_id"].str.extract(r"-R(\d+)")[0].astype(int)
    next_num = existing_nums.max() + 1
    return f"{athlete_id}-R{next_num:02d}"

## Load individual data, apply feature engineering

In [3]:
individuals_raw = pd.read_csv("../data/individuals/raw/individuals_raw.csv")
individuals_features = engineer_features(individuals_raw)

individuals_features.to_csv("../data/individuals/processed/individuals_features.csv", index=False)
individuals_features

,athlete_id,race_id,name,gender,height_cm,date_of_birth,race_date,final_time_sec,pb_50m_seconds,l1_reaction_time,l1_breakout_distance,l1_breakout_time,l1_stroke_count,l1_total_time,l2_breakout_distance,l2_breakout_time,l2_stroke_count,l2_total_time,l1_split_25m,l2_split_25m,age_at_race,l1_underwater_speed,l1_surface_distance,l1_surface_time,l1_surface_speed,l1_stroke_length,l1_relative_stroke_length,l1_stroke_rate,l1_stroke_index,l1_breakout_pct,l1_swolf,l2_underwater_speed,l2_surface_distance,l2_surface_time,l2_surface_speed,l2_stroke_length,l2_relative_stroke_length,l2_stroke_rate,l2_stroke_index,l2_breakout_pct,l2_swolf,l1_first25_speed,l1_second25_speed,intra_lap1_fade,intra_lap1_fade_ratio,l2_first25_speed,l2_second25_speed,intra_lap2_fade,intra_lap2_fade_ratio,finish_vs_fresh_ratio,pacing_decay_ratio,si_retention,breakout_decay_ratio,surface_speed_decay_ratio,front_end_pct,relative_stroke_length_drop,stroke_rate_change,swolf_change,breakout_drop
0,U001,U001-R01,Daniel Siahaan,male,168,2004-07-27,2021-09-02,57.53,26.5,0.67,12,4.6,37,27.81,6.5,2.7,45,29.72,12.3,14.2,17.100616,2.608696,38.0,23.21,1.637225,1.027027,0.611326,95.648427,1.681475,0.24,60.21,2.407407,43.5,27.02,1.609919,0.966667,0.575397,99.925981,1.556255,0.13,72.02,2.03252,1.611863,0.420657,0.793037,1.760563,1.610825,0.149739,0.914948,0.792526,1.06868,0.92553,0.541667,0.983321,1.049434,0.035929,4.277553,11.81,5.5


## Run full diagnosis: 6-Pillar report + lever-based simulation

Select a race by `race_id`. Two complementary outputs:
1. **6-Pillar report** — a full narrative picture across all six
   dimensions, each metric classified as Strength/On-par/Gap against the
   elite range for the swimmer's own gender.
2. **Lever-based simulation** — the SHAP-prioritized, optimizer-driven
   "what's the biggest actionable opportunity" view from Notebook 02's
   pipeline, giving a simulated performance range.

Neither output ranks the swimmer against anyone — everything is compared
to a range.

In [4]:
import optimization
print("simulate_underwater_speed_gap" in dir(optimization))

True


In [5]:
from optimization import simulate_underwater_speed_gap, print_gap_simulation

selected_race_id = "U001-R01"
user_row = individuals_features[individuals_features["race_id"] == selected_race_id].iloc[0]

# 1. Pillar report
pillar_report = compute_pillar_report(user_row, elite_features)
print_pillar_report(pillar_report)

biggest_gap = find_biggest_gap_pillar(pillar_report)
print(f"\nPrimary strategy focus (most Gap verdicts): {biggest_gap}")

print("\n" + "=" * 70 + "\n")

# 2. Gap-driven simulation — targets l1_underwater_speed, since that's
# the actual flagged Gap in the Underwater Hydrodynamics pillar
# (breakout_decay_ratio is now correctly On-par, not a weakness)
sim = simulate_underwater_speed_gap(user_row, elite_features, lap=1)
print_gap_simulation(sim)

Daniel Siahaan (male) — 6-Pillar Diagnostic Report

Pacing Decay
  pacing_decay_ratio: 1.069  [Strength]  (elite range: 1.066-1.134, mean 1.095)

Segmental Split Consistency
  intra_lap1_fade_ratio: 0.793  [Strength]  (elite range: 0.736-0.825, mean 0.773)
  intra_lap2_fade_ratio: 0.915  [Strength]  (elite range: 0.814-0.899, mean 0.859)
  finish_vs_fresh_ratio: 0.793  [Strength]  (elite range: 0.697-0.797, mean 0.740)

Underwater Hydrodynamics
  breakout_decay_ratio: 0.542  [On-par]  (elite range: 0.528-0.759, mean 0.641)
  l1_breakout_pct: 0.240  [On-par]  (elite range: 0.200-0.300, mean 0.259)
  l2_breakout_pct: 0.130  [Strength]  (elite range: 0.114-0.220, mean 0.166)
  l1_underwater_speed: 2.609  [Gap]  (elite range: 3.000-3.935, mean 3.272)
  l2_underwater_speed: 2.407  [On-par]  (elite range: 2.297-3.500, mean 2.727)

Stroke Mechanics & Water Grip
  si_retention: 0.926  [Strength]  (elite range: 0.813-0.995, mean 0.918)
  l1_relative_stroke_length: 0.611  [On-par]  (elite range: